In [2]:
print("Python Version:")
!python --version

print("\nEnvironment Details (installed packages):")
!pip list

Python Version:
Python 3.12.12

Environment Details (installed packages):
Package                                  Version
---------------------------------------- --------------------
absl-py                                  1.4.0
accelerate                               1.12.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.2
aiosignal                                1.4.0
aiosqlite                                0.22.0
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.17.2
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
an

# 🎯 PPT/PDF Extractor V6 - MAXIMUM ACCURACY

## Designed for Complex Business Presentations with Dense Data

**Focus: Extract EVERY piece of data accurately**
- ✅ Multiple extraction passes (OCR + VLM + Verification)
- ✅ Detailed prompts for charts, tables, metrics
- ✅ Cross-validation of numerical data
- ✅ Structured output with all data points
- ✅ Uses Qwen2-VL-7B for best accuracy

---
⚠️ **Enable GPU**: Runtime → Change runtime type → T4 GPU

In [1]:
%%time
# ═══════════════════════════════════════════════════════════════════════════════
# INSTALLATION
# ═══════════════════════════════════════════════════════════════════════════════

!pip uninstall -y paddlepaddle paddlepaddle-gpu paddleocr 2>/dev/null
!pip install paddlepaddle-gpu==2.6.2 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html -q
!pip install paddleocr==2.9.1 -q
!pip install -q pymupdf Pillow tqdm
!pip install -q transformers accelerate qwen-vl-utils bitsandbytes
!pip install -q opencv-python-headless numpy tabulate ipywidgets

print("\n" + "="*60)
print("✅ All dependencies installed!")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.9/758.9 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.7/544.7 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.9/161.9 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════

import torch
import gc
import os
import io
import json
import re
import cv2
import numpy as np
import fitz
from PIL import Image
from tqdm.notebook import tqdm
from typing import List, Dict, Optional, Tuple, Any
from dataclasses import dataclass, field
from datetime import datetime
from collections import defaultdict, OrderedDict
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from io import BytesIO
import base64
import warnings
warnings.filterwarnings('ignore')

# GPU Check
print("="*60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("❌ NO GPU")
print("="*60)

✅ GPU: Tesla T4 (14.7 GB)


In [1]:
%%time
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD PP-OCR ENGINE
# ═══════════════════════════════════════════════════════════════════════════════

from paddleocr import PaddleOCR, PPStructure

print("🔄 Loading PP-OCR...")

ppocr_engine = PaddleOCR(
    use_angle_cls=True,
    lang='en',
    use_gpu=True,
    show_log=False,
    ocr_version='PP-OCRv4',
    det_db_thresh=0.3,      # Lower threshold for better detection
    det_db_box_thresh=0.5,
)

pp_structure = PPStructure(
    table=True,
    ocr=True,
    show_log=False,
    use_gpu=True,
    layout=True
)

print("✅ PP-OCR Ready")

🔄 Loading PP-OCR...
download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 3910/3910 [00:00<00:00, 4617.16it/s]


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10000/10000 [00:00<00:00, 12849.47it/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


...

download https://paddleocr.bj.bcebos.com/ppstructure/models/layout/picodet_lcnet_x1_0_fgd_layout_cdla_infer.tar to /root/.paddleocr/whl/layout/picodet_lcnet_x1_0_fgd_layout_cdla_infer/picodet_lcnet_x1_0_fgd_layout_cdla_infer.tar


100%|██████████| 9870/9870 [00:01<00:00, 7416.05it/s] 


download https://paddleocr.bj.bcebos.com/contribution/rec_latex_ocr_infer.tar to /root/.paddleocr/whl/formula/rec_latex_ocr_infer/rec_latex_ocr_infer.tar


100%|██████████| 104000/104000 [00:06<00:00, 16047.50it/s]


[2026/01/02 17:47:11] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
[2026/01/02 17:47:13] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
[2026/01/02 17:47:14] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
[2026/01/02 17:47:15] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
[2026/01/02 17:47:16] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
✅ PP-OCR Ready
CPU times: user 17.3 s, sys: 1.66 s, total: 19 s
Wall time: 48 s


In [4]:
%%time
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD VLM - Qwen2-VL-7B for MAXIMUM ACCURACY
# ═══════════════════════════════════════════════════════════════════════════════

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

VLM_MODEL = "Qwen/Qwen2-VL-7B-Instruct"

print(f"🔄 Loading {VLM_MODEL} (this takes a few minutes)...")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
    VLM_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL, trust_remote_code=True)

print(f"✅ VLM Ready | GPU: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

🔄 Loading Qwen/Qwen2-VL-7B-Instruct (this takes a few minutes)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

✅ VLM Ready | GPU: 5.53 GB
CPU times: user 1min 28s, sys: 54.9 s, total: 2min 23s
Wall time: 4min 8s


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# COMPREHENSIVE OCR EXTRACTOR
# ═══════════════════════════════════════════════════════════════════════════════

class ComprehensiveOCRExtractor:
    """Extract ALL text, numbers, tables with maximum precision"""

    def __init__(self, ocr_engine, structure_engine):
        self.ocr = ocr_engine
        self.structure = structure_engine

    def extract(self, image: np.ndarray) -> Dict:
        """Complete OCR extraction with multiple passes"""

        result = {
            'text_blocks': [],
            'tables': [],
            'layout_regions': [],
            'all_text': [],
            'all_numbers': [],
            'percentages': [],
            'currency_values': [],
            'years': [],
            'raw_text': '',
            'confidence': 0.0
        }

        # Pass 1: Standard OCR
        try:
            ocr_result = self.ocr.ocr(image, cls=True)

            if ocr_result and ocr_result[0]:
                confidences = []
                for line in ocr_result[0]:
                    bbox = line[0]
                    text = line[1][0]
                    conf = float(line[1][1])

                    # Calculate position for sorting
                    center_y = sum(p[1] for p in bbox) / 4
                    center_x = sum(p[0] for p in bbox) / 4

                    result['text_blocks'].append({
                        'text': text,
                        'bbox': bbox,
                        'confidence': conf,
                        'center_x': center_x,
                        'center_y': center_y
                    })
                    result['all_text'].append(text)
                    confidences.append(conf)

                    # Extract ALL numbers
                    self._extract_numbers(text, result)

                result['confidence'] = sum(confidences) / len(confidences) if confidences else 0
        except Exception as e:
            print(f"   ⚠️ OCR Pass 1 error: {e}")

        # Pass 2: Structure analysis
        try:
            structure_result = self.structure(image)

            for item in structure_result:
                item_type = item.get('type', 'unknown')

                result['layout_regions'].append({
                    'type': item_type,
                    'bbox': item.get('bbox', []),
                    'score': item.get('score', 0.0)
                })

                if item_type == 'table':
                    table_data = self._parse_table(item)
                    if table_data:
                        result['tables'].append(table_data)
                        # Extract numbers from table
                        for row in table_data.get('all_data', []):
                            for cell in row:
                                self._extract_numbers(str(cell), result)
        except Exception as e:
            print(f"   ⚠️ Structure Pass error: {e}")

        # Sort text by position (top-to-bottom, left-to-right)
        result['text_blocks'].sort(key=lambda x: (x['center_y'], x['center_x']))
        result['raw_text'] = ' '.join([b['text'] for b in result['text_blocks']])

        # Remove duplicates from number lists
        result['all_numbers'] = list(OrderedDict.fromkeys(result['all_numbers']))
        result['percentages'] = list(OrderedDict.fromkeys(result['percentages']))
        result['currency_values'] = list(OrderedDict.fromkeys(result['currency_values']))
        result['years'] = list(OrderedDict.fromkeys(result['years']))

        return result

    def _extract_numbers(self, text: str, result: Dict):
        """Extract all numerical values from text"""

        # Percentages: 6.7%, 13%, etc.
        percentages = re.findall(r'[\d,]+\.?\d*\s*%', text)
        result['percentages'].extend(percentages)

        # Currency: $500, $1.2Bn, $35 Tn, etc.
        currencies = re.findall(r'\$[\d,]+\.?\d*\s*[BMKTn]*\+?', text, re.IGNORECASE)
        result['currency_values'].extend(currencies)

        # Years: FY25, FY32E, 2024, 2047, etc.
        years = re.findall(r"FY\d{2}E?|20\d{2}|'\d{2}", text)
        result['years'].extend(years)

        # General numbers with units: 1.46, 186 bn, 112 GW+, etc.
        numbers = re.findall(r'[\d,]+\.?\d*\s*(?:bn|Bn|BN|mn|Mn|MN|GW|MW|MWh|ckms|x|Tn|TN)?\+?', text)
        result['all_numbers'].extend([n.strip() for n in numbers if n.strip()])

    def _parse_table(self, table_item: Dict) -> Optional[Dict]:
        """Parse table with all data"""
        try:
            res = table_item.get('res', {})
            html = res.get('html', '')

            if not html:
                return None

            from html.parser import HTMLParser

            class TableParser(HTMLParser):
                def __init__(self):
                    super().__init__()
                    self.rows = []
                    self.current_row = []
                    self.current_cell = ""
                    self.in_cell = False

                def handle_starttag(self, tag, attrs):
                    if tag in ['td', 'th']:
                        self.in_cell = True
                        self.current_cell = ""

                def handle_endtag(self, tag):
                    if tag in ['td', 'th']:
                        self.in_cell = False
                        self.current_row.append(self.current_cell.strip())
                    elif tag == 'tr':
                        if self.current_row:
                            self.rows.append(self.current_row)
                        self.current_row = []

                def handle_data(self, data):
                    if self.in_cell:
                        self.current_cell += data

            parser = TableParser()
            parser.feed(html)

            if parser.rows:
                return {
                    'headers': parser.rows[0] if parser.rows else [],
                    'rows': parser.rows[1:] if len(parser.rows) > 1 else [],
                    'all_data': parser.rows,
                    'num_rows': len(parser.rows),
                    'num_cols': len(parser.rows[0]) if parser.rows else 0
                }
            return None
        except:
            return None

ocr_extractor = ComprehensiveOCRExtractor(ppocr_engine, pp_structure)
print("✅ OCR Extractor Ready")

✅ OCR Extractor Ready


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DETAILED VLM EXTRACTION PROMPTS
# ═══════════════════════════════════════════════════════════════════════════════

# Main comprehensive prompt
MAIN_EXTRACTION_PROMPT = '''You are an expert data analyst. Analyze this presentation slide and extract ALL information.

CRITICAL: Extract EVERY number, percentage, currency value, and data point visible.

Return a JSON object with this EXACT structure:
{
    "slide_title": "exact title of the slide",
    "slide_type": "data/chart/table/mixed/title",
    "company_brand": "company name if visible",

    "key_highlights": [
        "bullet point 1 with exact numbers",
        "bullet point 2 with exact numbers"
    ],

    "metrics": {
        "metric_name": {"value": "exact value", "unit": "unit", "context": "what it represents"}
    },

    "bar_charts": [
        {
            "title": "chart title",
            "x_axis": "what x-axis shows",
            "y_axis": "what y-axis shows (with unit)",
            "data_points": {"label1": "value1", "label2": "value2"},
            "insight": "what the chart shows"
        }
    ],

    "line_charts": [
        {
            "title": "chart title",
            "data_series": {"year/period": "value"},
            "trend": "increasing/decreasing/stable",
            "insight": "what it shows"
        }
    ],

    "tables": [
        {
            "title": "table title",
            "headers": ["col1", "col2"],
            "rows": [["row1col1", "row1col2"], ["row2col1", "row2col2"]],
            "summary": "key takeaway from table"
        }
    ],

    "comparisons": [
        {"entity1": "name", "entity1_value": "value", "entity2": "name", "entity2_value": "value", "metric": "what is compared"}
    ],

    "growth_rates": [
        {"metric": "what is growing", "rate": "growth rate", "period": "time period", "from": "start value", "to": "end value"}
    ],

    "targets_projections": [
        {"target": "what", "value": "target value", "by_when": "deadline/year"}
    ],

    "country_data": {
        "country_name": {"metric": "value"}
    },

    "investment_opportunities": [
        {"sector": "sector name", "amount": "investment amount", "timeline": "by when"}
    ],

    "all_numerical_facts": [
        "India GDP growth: 6.5% in FY25",
        "Population: 1.46 billion"
    ],

    "sources": ["source 1", "source 2"]
}

IMPORTANT RULES:
1. Extract EVERY visible number, percentage, and currency value
2. Include units (%, GW, Bn, $, etc.)
3. Preserve exact values - do not round or estimate
4. For charts, extract ALL data points visible
5. For tables, extract ALL rows and columns
6. Return ONLY valid JSON, no other text'''

# Focused chart extraction prompt
CHART_EXTRACTION_PROMPT = '''Extract ALL data from the charts in this slide.

For EACH chart visible, provide:
{
    "charts": [
        {
            "chart_number": 1,
            "chart_type": "bar/line/pie/area",
            "title": "exact chart title",
            "subtitle": "subtitle if any",
            "x_axis_label": "label",
            "y_axis_label": "label with unit",
            "data": {
                "category1": "value1",
                "category2": "value2"
            },
            "annotations": ["any text annotations on chart"],
            "insight": "what the chart communicates"
        }
    ]
}

Extract EVERY data point. Return ONLY JSON.'''

# Focused table extraction prompt
TABLE_EXTRACTION_PROMPT = '''Extract ALL data from tables in this slide.

For EACH table:
{
    "tables": [
        {
            "table_number": 1,
            "title": "table title",
            "headers": ["header1", "header2", "header3"],
            "data_rows": [
                ["row1_col1", "row1_col2", "row1_col3"],
                ["row2_col1", "row2_col2", "row2_col3"]
            ],
            "totals_row": ["total values if present"],
            "key_values": {"row_label": "important_value"}
        }
    ]
}

Include ALL rows and columns. Return ONLY JSON.'''

# Numbers verification prompt
NUMBERS_VERIFICATION_PROMPT = '''List EVERY number visible in this slide.

Return JSON:
{
    "percentages": ["6.7%", "6.4%"],
    "currency_amounts": ["$500 Bn+", "$35 Tn"],
    "quantities": ["1.46 Bn", "186 bn"],
    "growth_rates": ["11% CAGR", "14x increase"],
    "years": ["FY25", "FY32", "2047"],
    "other_numbers": ["112 GW+", "648 ckms"]
}

Include EVERY number. Return ONLY JSON.'''

print("✅ Extraction Prompts Defined")

✅ Extraction Prompts Defined


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MULTI-PASS VLM EXTRACTOR
# ═══════════════════════════════════════════════════════════════════════════════

class MultiPassVLMExtractor:
    """Extract with multiple focused passes for maximum accuracy"""

    def __init__(self, model, processor):
        self.model = model
        self.processor = processor
        self.max_tokens = 4096

    def _run_vlm(self, image: Image.Image, prompt: str) -> str:
        """Run VLM with given prompt"""
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        image_inputs, video_inputs = process_vision_info(messages)

        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_tokens,
                do_sample=False,
            )

        response = self.processor.batch_decode(
            outputs[:, inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )[0]

        return response

    def _parse_json(self, text: str) -> Dict:
        """Robust JSON parsing"""
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0]
        elif "```" in text:
            parts = text.split("```")
            if len(parts) >= 2:
                text = parts[1]

        text = text.strip()

        start = text.find('{')
        end = text.rfind('}') + 1

        if start >= 0 and end > start:
            json_str = text[start:end]
            # Fix common JSON issues
            json_str = re.sub(r',\s*}', '}', json_str)
            json_str = re.sub(r',\s*]', ']', json_str)
            json_str = re.sub(r'\n', ' ', json_str)

            try:
                return json.loads(json_str)
            except json.JSONDecodeError:
                # Try to fix more issues
                json_str = re.sub(r"'([^']*)':", r'"\1":', json_str)
                try:
                    return json.loads(json_str)
                except:
                    pass

        return {'raw_response': text, 'parse_error': True}

    def extract_comprehensive(self, image: Image.Image, ocr_context: str = "") -> Dict:
        """Multi-pass extraction for maximum accuracy"""

        results = {
            'main_extraction': {},
            'chart_extraction': {},
            'table_extraction': {},
            'numbers_verification': {},
            'combined': {}
        }

        # Add OCR context to main prompt
        main_prompt = MAIN_EXTRACTION_PROMPT
        if ocr_context:
            main_prompt += f"\n\nOCR detected text for reference:\n{ocr_context[:2000]}"

        # Pass 1: Main comprehensive extraction
        print("      → Pass 1: Main extraction...")
        try:
            response = self._run_vlm(image, main_prompt)
            results['main_extraction'] = self._parse_json(response)
        except Exception as e:
            print(f"      ⚠️ Main extraction error: {str(e)[:50]}")

        gc.collect()
        torch.cuda.empty_cache()

        # Pass 2: Focused chart extraction
        print("      → Pass 2: Chart extraction...")
        try:
            response = self._run_vlm(image, CHART_EXTRACTION_PROMPT)
            results['chart_extraction'] = self._parse_json(response)
        except Exception as e:
            print(f"      ⚠️ Chart extraction error: {str(e)[:50]}")

        gc.collect()
        torch.cuda.empty_cache()

        # Pass 3: Focused table extraction
        print("      → Pass 3: Table extraction...")
        try:
            response = self._run_vlm(image, TABLE_EXTRACTION_PROMPT)
            results['table_extraction'] = self._parse_json(response)
        except Exception as e:
            print(f"      ⚠️ Table extraction error: {str(e)[:50]}")

        gc.collect()
        torch.cuda.empty_cache()

        # Pass 4: Numbers verification
        print("      → Pass 4: Numbers verification...")
        try:
            response = self._run_vlm(image, NUMBERS_VERIFICATION_PROMPT)
            results['numbers_verification'] = self._parse_json(response)
        except Exception as e:
            print(f"      ⚠️ Numbers verification error: {str(e)[:50]}")

        # Combine all results
        results['combined'] = self._combine_results(results)

        return results

    def _combine_results(self, results: Dict) -> Dict:
        """Combine results from all passes"""
        combined = {}

        main = results.get('main_extraction', {})
        charts = results.get('chart_extraction', {})
        tables = results.get('table_extraction', {})
        numbers = results.get('numbers_verification', {})

        # Basic info from main
        combined['slide_title'] = main.get('slide_title', '')
        combined['slide_type'] = main.get('slide_type', 'unknown')
        combined['company_brand'] = main.get('company_brand', '')
        combined['key_highlights'] = main.get('key_highlights', [])
        combined['metrics'] = main.get('metrics', {})
        combined['sources'] = main.get('sources', [])

        # Charts - prefer focused extraction
        combined['charts'] = charts.get('charts', []) or main.get('bar_charts', []) + main.get('line_charts', [])

        # Tables - prefer focused extraction
        combined['tables'] = tables.get('tables', []) or main.get('tables', [])

        # Comparisons and growth
        combined['comparisons'] = main.get('comparisons', [])
        combined['growth_rates'] = main.get('growth_rates', [])
        combined['targets_projections'] = main.get('targets_projections', [])
        combined['investment_opportunities'] = main.get('investment_opportunities', [])
        combined['country_data'] = main.get('country_data', {})

        # Numbers - combine from verification and main
        combined['verified_numbers'] = {
            'percentages': numbers.get('percentages', []),
            'currency_amounts': numbers.get('currency_amounts', []),
            'quantities': numbers.get('quantities', []),
            'growth_rates': numbers.get('growth_rates', []),
            'years': numbers.get('years', []),
            'other_numbers': numbers.get('other_numbers', [])
        }

        # All numerical facts
        combined['all_numerical_facts'] = main.get('all_numerical_facts', [])

        return combined

vlm_extractor = MultiPassVLMExtractor(vlm_model, vlm_processor)
print("✅ Multi-Pass VLM Extractor Ready")

✅ Multi-Pass VLM Extractor Ready


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# INTELLIGENT DATA FUSION
# ═══════════════════════════════════════════════════════════════════════════════

class IntelligentDataFusion:
    """Merge OCR and VLM results with cross-validation"""

    def fuse(self, page_num: int, ocr_result: Dict, vlm_result: Dict) -> Dict:
        """Combine all extraction results"""

        vlm_combined = vlm_result.get('combined', {})

        fused = {
            'page_number': page_num,

            # Slide info (from VLM)
            'slide_title': vlm_combined.get('slide_title', ''),
            'slide_type': vlm_combined.get('slide_type', 'unknown'),
            'company_brand': vlm_combined.get('company_brand', ''),

            # Key content (from VLM)
            'key_highlights': vlm_combined.get('key_highlights', []),
            'metrics': vlm_combined.get('metrics', {}),

            # Charts (from VLM - semantic understanding)
            'charts': vlm_combined.get('charts', []),

            # Tables - combine OCR and VLM
            'tables_ocr': ocr_result.get('tables', []),
            'tables_vlm': vlm_combined.get('tables', []),

            # Comparisons and analysis (from VLM)
            'comparisons': vlm_combined.get('comparisons', []),
            'growth_rates': vlm_combined.get('growth_rates', []),
            'targets_projections': vlm_combined.get('targets_projections', []),
            'investment_opportunities': vlm_combined.get('investment_opportunities', []),
            'country_data': vlm_combined.get('country_data', {}),

            # Numbers - combine and cross-validate
            'numbers_ocr': {
                'all': ocr_result.get('all_numbers', []),
                'percentages': ocr_result.get('percentages', []),
                'currency': ocr_result.get('currency_values', []),
                'years': ocr_result.get('years', [])
            },
            'numbers_vlm': vlm_combined.get('verified_numbers', {}),

            # All text (from OCR - accurate)
            'all_text_ocr': ocr_result.get('all_text', []),
            'raw_text': ocr_result.get('raw_text', ''),

            # Numerical facts (from VLM)
            'numerical_facts': vlm_combined.get('all_numerical_facts', []),

            # Layout (from OCR structure)
            'layout_regions': ocr_result.get('layout_regions', []),

            # Sources
            'sources': vlm_combined.get('sources', []),

            # Confidence
            'ocr_confidence': ocr_result.get('confidence', 0),

            # Raw VLM results for debugging
            'vlm_raw': {
                'main': vlm_result.get('main_extraction', {}),
                'charts': vlm_result.get('chart_extraction', {}),
                'tables': vlm_result.get('table_extraction', {}),
                'numbers': vlm_result.get('numbers_verification', {})
            }
        }

        # Generate comprehensive factual statements
        fused['factual_statements'] = self._generate_factual_statements(fused)

        # Cross-validate numbers
        fused['validated_numbers'] = self._cross_validate_numbers(fused)

        # Assess extraction quality
        fused['extraction_quality'] = self._assess_quality(fused)

        return fused

    def _generate_factual_statements(self, fused: Dict) -> List[str]:
        """Generate all factual statements from extracted data"""
        facts = []

        # From key highlights
        facts.extend(fused.get('key_highlights', []))

        # From metrics
        for name, data in fused.get('metrics', {}).items():
            if isinstance(data, dict):
                value = data.get('value', '')
                unit = data.get('unit', '')
                context = data.get('context', '')
                facts.append(f"{name}: {value} {unit} - {context}")
            else:
                facts.append(f"{name}: {data}")

        # From charts
        for chart in fused.get('charts', []):
            if isinstance(chart, dict):
                title = chart.get('title', '')
                if title:
                    facts.append(f"Chart: {title}")
                for label, value in chart.get('data', {}).items():
                    facts.append(f"  {label}: {value}")
                for label, value in chart.get('data_points', {}).items():
                    facts.append(f"  {label}: {value}")
                if chart.get('insight'):
                    facts.append(f"  Insight: {chart['insight']}")

        # From tables
        for table in fused.get('tables_vlm', []):
            if isinstance(table, dict):
                if table.get('title'):
                    facts.append(f"Table: {table['title']}")
                for row in table.get('data_rows', table.get('rows', []))[:10]:
                    if row:
                        facts.append(f"  {' | '.join(str(c) for c in row)}")

        # From comparisons
        for comp in fused.get('comparisons', []):
            if isinstance(comp, dict):
                e1 = comp.get('entity1', '')
                v1 = comp.get('entity1_value', '')
                e2 = comp.get('entity2', '')
                v2 = comp.get('entity2_value', '')
                metric = comp.get('metric', '')
                facts.append(f"Comparison ({metric}): {e1}={v1} vs {e2}={v2}")

        # From growth rates
        for growth in fused.get('growth_rates', []):
            if isinstance(growth, dict):
                facts.append(f"Growth: {growth.get('metric', '')} - {growth.get('rate', '')} ({growth.get('period', '')})")

        # From targets
        for target in fused.get('targets_projections', []):
            if isinstance(target, dict):
                facts.append(f"Target: {target.get('target', '')} = {target.get('value', '')} by {target.get('by_when', '')}")

        # From investments
        for inv in fused.get('investment_opportunities', []):
            if isinstance(inv, dict):
                facts.append(f"Investment: {inv.get('sector', '')} - {inv.get('amount', '')} by {inv.get('timeline', '')}")

        # From country data
        for country, data in fused.get('country_data', {}).items():
            if isinstance(data, dict):
                for metric, value in data.items():
                    facts.append(f"{country} - {metric}: {value}")

        # From VLM numerical facts
        facts.extend(fused.get('numerical_facts', []))

        # Clean and dedupe
        clean_facts = []
        seen = set()
        for f in facts:
            if f and f.strip() and f.strip() not in seen:
                clean_facts.append(f.strip())
                seen.add(f.strip())

        return clean_facts

    def _cross_validate_numbers(self, fused: Dict) -> Dict:
        """Cross-validate numbers from OCR and VLM"""

        ocr_numbers = set(fused.get('numbers_ocr', {}).get('all', []))
        vlm_numbers = set()

        vlm_num_data = fused.get('numbers_vlm', {})
        for key in ['percentages', 'currency_amounts', 'quantities', 'growth_rates', 'other_numbers']:
            vlm_numbers.update(vlm_num_data.get(key, []))

        # Find overlapping (validated) numbers
        validated = ocr_numbers.intersection(vlm_numbers)

        return {
            'ocr_only': list(ocr_numbers - vlm_numbers),
            'vlm_only': list(vlm_numbers - ocr_numbers),
            'validated_both': list(validated),
            'all_unique': list(ocr_numbers.union(vlm_numbers))
        }

    def _assess_quality(self, fused: Dict) -> str:
        """Assess extraction quality"""
        score = 0

        if fused.get('slide_title'):
            score += 1
        if len(fused.get('key_highlights', [])) > 0:
            score += 2
        if len(fused.get('metrics', {})) > 0:
            score += 2
        if len(fused.get('charts', [])) > 0:
            score += 2
        if len(fused.get('tables_vlm', [])) > 0 or len(fused.get('tables_ocr', [])) > 0:
            score += 2
        if len(fused.get('factual_statements', [])) > 5:
            score += 2
        if len(fused.get('validated_numbers', {}).get('all_unique', [])) > 10:
            score += 2
        if fused.get('ocr_confidence', 0) > 0.8:
            score += 1

        if score >= 10:
            return 'excellent'
        elif score >= 7:
            return 'good'
        elif score >= 4:
            return 'fair'
        else:
            return 'poor'

fusion_engine = IntelligentDataFusion()
print("✅ Data Fusion Engine Ready")

✅ Data Fusion Engine Ready


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

class MaxAccuracyPipeline:
    """Maximum accuracy extraction pipeline"""

    def __init__(self, ocr_extractor, vlm_extractor, fusion_engine):
        self.ocr = ocr_extractor
        self.vlm = vlm_extractor
        self.fusion = fusion_engine

    def pdf_to_images(self, pdf_path: str, zoom: float = 2.5,
                      start_page: int = 0, end_page: int = None) -> List[Tuple]:
        """Convert PDF to high-res images"""
        doc = fitz.open(pdf_path)
        end = end_page if end_page else doc.page_count
        images = []

        for i in tqdm(range(start_page, end), desc="PDF → Images"):
            page = doc[i]
            mat = fitz.Matrix(zoom, zoom)
            pix = page.get_pixmap(matrix=mat, alpha=False)
            pil_img = Image.open(io.BytesIO(pix.tobytes("png")))
            np_img = np.array(pil_img)
            if len(np_img.shape) == 2:
                np_img = cv2.cvtColor(np_img, cv2.COLOR_GRAY2RGB)
            images.append((i + 1, pil_img, np_img))

        doc.close()
        return images

    def process_slide(self, page_num: int, pil_img: Image.Image, np_img: np.ndarray) -> Dict:
        """Process single slide with maximum accuracy"""
        import time
        start = time.time()

        print(f"\n{'═'*60}")
        print(f"📄 PROCESSING SLIDE {page_num}")
        print(f"{'═'*60}")

        # Step 1: Comprehensive OCR
        print("\n🔤 Step 1: OCR Extraction")
        ocr_result = self.ocr.extract(np_img)
        print(f"   ✓ Text blocks: {len(ocr_result['all_text'])}")
        print(f"   ✓ Tables detected: {len(ocr_result['tables'])}")
        print(f"   ✓ Numbers found: {len(ocr_result['all_numbers'])}")
        print(f"   ✓ Percentages: {len(ocr_result['percentages'])}")
        print(f"   ✓ Currency values: {len(ocr_result['currency_values'])}")
        print(f"   ✓ Confidence: {ocr_result['confidence']:.2%}")

        # Step 2: Multi-pass VLM extraction
        print("\n🧠 Step 2: VLM Multi-Pass Extraction")
        vlm_result = self.vlm.extract_comprehensive(pil_img, ocr_result['raw_text'])

        vlm_combined = vlm_result.get('combined', {})
        print(f"   ✓ Title: {vlm_combined.get('slide_title', 'N/A')[:60]}")
        print(f"   ✓ Type: {vlm_combined.get('slide_type', 'N/A')}")
        print(f"   ✓ Key highlights: {len(vlm_combined.get('key_highlights', []))}")
        print(f"   ✓ Metrics: {len(vlm_combined.get('metrics', {}))}")
        print(f"   ✓ Charts: {len(vlm_combined.get('charts', []))}")
        print(f"   ✓ Tables: {len(vlm_combined.get('tables', []))}")

        # Step 3: Intelligent fusion
        print("\n🔀 Step 3: Data Fusion")
        fused = self.fusion.fuse(page_num, ocr_result, vlm_result)
        print(f"   ✓ Factual statements: {len(fused['factual_statements'])}")
        print(f"   ✓ Validated numbers: {len(fused['validated_numbers'].get('all_unique', []))}")
        print(f"   ✓ Quality: {fused['extraction_quality'].upper()}")

        elapsed = time.time() - start
        print(f"\n⏱️  Completed in {elapsed:.1f}s")

        gc.collect()
        torch.cuda.empty_cache()

        return fused

    def process_document(self, pdf_path: str, zoom: float = 2.5,
                         start_page: int = 0, end_page: int = None) -> Tuple[Dict, Dict]:
        """Process entire document"""
        import time

        print("\n" + "═"*70)
        print("🎯 MAXIMUM ACCURACY EXTRACTION PIPELINE")
        print("═"*70)
        print(f"📁 Document: {os.path.basename(pdf_path)}")
        print(f"🔧 Zoom: {zoom}x | Multi-pass VLM | Cross-validation enabled")
        print("═"*70)

        start_time = time.time()

        slides = self.pdf_to_images(pdf_path, zoom, start_page, end_page)
        print(f"\n✅ Converted {len(slides)} slides to high-res images")

        results = []
        stored_images = {}

        for page_num, pil_img, np_img in slides:
            stored_images[page_num] = pil_img.copy()
            result = self.process_slide(page_num, pil_img, np_img)
            results.append(result)

        total_time = time.time() - start_time

        output = {
            'document_info': {
                'source_file': os.path.basename(pdf_path),
                'total_pages': len(slides),
                'processing_time_seconds': total_time,
                'timestamp': datetime.now().isoformat(),
                'pipeline_version': 'V6-MaxAccuracy'
            },
            'pages': results
        }

        self._print_summary(output)

        return output, stored_images

    def _print_summary(self, output: Dict):
        """Print comprehensive summary"""
        print("\n" + "═"*70)
        print("📊 EXTRACTION SUMMARY")
        print("═"*70)

        total_facts = 0
        total_numbers = 0
        total_charts = 0
        total_tables = 0

        for page in output['pages']:
            total_facts += len(page.get('factual_statements', []))
            total_numbers += len(page.get('validated_numbers', {}).get('all_unique', []))
            total_charts += len(page.get('charts', []))
            total_tables += len(page.get('tables_vlm', [])) + len(page.get('tables_ocr', []))

        print(f"📁 Document: {output['document_info']['source_file']}")
        print(f"📄 Pages processed: {output['document_info']['total_pages']}")
        print(f"⏱️  Total time: {output['document_info']['processing_time_seconds']:.1f}s")
        print(f"\n📈 Extracted Data:")
        print(f"   • Factual statements: {total_facts}")
        print(f"   • Unique numbers: {total_numbers}")
        print(f"   • Charts analyzed: {total_charts}")
        print(f"   • Tables detected: {total_tables}")

        qualities = [p.get('extraction_quality', 'unknown') for p in output['pages']]
        print(f"\n📊 Quality:")
        for q in ['excellent', 'good', 'fair', 'poor']:
            count = qualities.count(q)
            if count > 0:
                print(f"   • {q.capitalize()}: {count} slides")

        print("═"*70)

pipeline = MaxAccuracyPipeline(ocr_extractor, vlm_extractor, fusion_engine)
print("\n" + "═"*70)
print("✅ MAXIMUM ACCURACY PIPELINE READY")
print("═"*70)


══════════════════════════════════════════════════════════════════════
✅ MAXIMUM ACCURACY PIPELINE READY
══════════════════════════════════════════════════════════════════════


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# UPLOAD PDF
# ═══════════════════════════════════════════════════════════════════════════════

from google.colab import files

print("📁 Upload your PDF file:")
uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {pdf_file}")

📁 Upload your PDF file:


Saving test (1).pdf to test (1).pdf

✅ Uploaded: test (1).pdf


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# RUN EXTRACTION
# ═══════════════════════════════════════════════════════════════════════════════

# Configuration - Higher zoom for better accuracy
ZOOM = 2.5          # Higher zoom = better detail
START_PAGE = 0      # 0-indexed
END_PAGE = None     # None = all pages

# Run extraction
results, stored_images = pipeline.process_document(
    pdf_file,
    zoom=ZOOM,
    start_page=START_PAGE,
    end_page=END_PAGE
)


══════════════════════════════════════════════════════════════════════
🎯 MAXIMUM ACCURACY EXTRACTION PIPELINE
══════════════════════════════════════════════════════════════════════
📁 Document: test (1).pdf
🔧 Zoom: 2.5x | Multi-pass VLM | Cross-validation enabled
══════════════════════════════════════════════════════════════════════


PDF → Images:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Converted 2 slides to high-res images

════════════════════════════════════════════════════════════
📄 PROCESSING SLIDE 1
════════════════════════════════════════════════════════════

🔤 Step 1: OCR Extraction
   ⚠️ Structure Pass error: 

--------------------------------------
C++ Traceback (most recent call last):
--------------------------------------
0   paddle::AnalysisPredictor::ZeroCopyRun()
1   paddle::framework::NaiveExecutor::Run()
2   paddle::framework::OperatorBase::Run(paddle::framework::Scope const&, phi::Place const&)
3   paddle::framework::OperatorWithKernel::RunImpl(paddle::framework::Scope const&, phi::Place const&) const
4   paddle::framework::OperatorWithKernel::RunImpl(paddle::framework::Scope const&, phi::Place const&, paddle::framework::RuntimeContext*) const
5   void phi::MultiplyRawKernel<float, phi::GPUContext>(phi::GPUContext const&, phi::DenseTensor const&, phi::DenseTensor const&, int, phi::DenseTensor*)
6   float* phi::DeviceContext::Alloc<float>(phi::Ten

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


      ⚠️ Main extraction error: CUDA out of memory. Tried to allocate 3.07 GiB. GP
      → Pass 2: Chart extraction...
      ⚠️ Chart extraction error: CUDA out of memory. Tried to allocate 1.93 GiB. GP
      → Pass 3: Table extraction...
      ⚠️ Table extraction error: CUDA out of memory. Tried to allocate 1.92 GiB. GP
      → Pass 4: Numbers verification...
      ⚠️ Numbers verification error: CUDA out of memory. Tried to allocate 1.91 GiB. GP
   ✓ Title: 
   ✓ Type: unknown
   ✓ Key highlights: 0
   ✓ Metrics: 0
   ✓ Charts: 0
   ✓ Tables: 0

🔀 Step 3: Data Fusion
   ✓ Factual statements: 0
   ✓ Validated numbers: 72
   ✓ Quality: POOR

⏱️  Completed in 46.8s

════════════════════════════════════════════════════════════
📄 PROCESSING SLIDE 2
════════════════════════════════════════════════════════════

🔤 Step 1: OCR Extraction
   ⚠️ OCR Pass 1 error: 

--------------------------------------
C++ Traceback (most recent call last):
--------------------------------------
0   void paddle

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# COMPREHENSIVE RESULTS VIEWER
# ═══════════════════════════════════════════════════════════════════════════════

def display_comprehensive_results(page_num: int):
    """Display ALL extracted data for a slide"""

    page_data = None
    for p in results['pages']:
        if p.get('page_number') == page_num:
            page_data = p
            break

    if not page_data:
        print(f"❌ Page {page_num} not found")
        return

    # Image
    if page_num in stored_images:
        img = stored_images[page_num]
        ratio = 700 / img.width
        new_size = (700, int(img.height * ratio))
        img_resized = img.resize(new_size, Image.LANCZOS)
        buffer = BytesIO()
        img_resized.save(buffer, format="PNG")
        img_b64 = base64.b64encode(buffer.getvalue()).decode()
    else:
        img_b64 = None

    html = f"""
    <div style="font-family: Arial; max-width: 1400px; padding: 20px;">
        <h1 style="color: #1a73e8;">📄 Slide {page_num}: {page_data.get('slide_title', 'Untitled')}</h1>
        <p style="font-size: 14px; color: #666;">
            <b>Type:</b> {page_data.get('slide_type', 'N/A')} |
            <b>Brand:</b> {page_data.get('company_brand', 'N/A')} |
            <b>Quality:</b> <span style="color: {'green' if page_data.get('extraction_quality') == 'excellent' else 'orange'};">
                {page_data.get('extraction_quality', 'N/A').upper()}</span> |
            <b>OCR Confidence:</b> {page_data.get('ocr_confidence', 0):.1%}
        </p>
    """

    if img_b64:
        html += f'<img src="data:image/png;base64,{img_b64}" style="max-width:700px; border:2px solid #ddd; border-radius:8px;"><br><br>'

    # Key Highlights
    if page_data.get('key_highlights'):
        html += "<h2 style='color: #34a853;'>📌 Key Highlights</h2><ul>"
        for h in page_data['key_highlights']:
            html += f"<li style='margin: 8px 0;'>{h}</li>"
        html += "</ul>"

    # Metrics
    if page_data.get('metrics'):
        html += "<h2 style='color: #ea4335;'>📊 Metrics</h2>"
        html += "<table style='border-collapse: collapse; width: 100%;'>"
        html += "<tr style='background: #f5f5f5;'><th style='padding:10px; border:1px solid #ddd;'>Metric</th><th style='padding:10px; border:1px solid #ddd;'>Value</th><th style='padding:10px; border:1px solid #ddd;'>Context</th></tr>"
        for name, data in page_data['metrics'].items():
            if isinstance(data, dict):
                html += f"<tr><td style='padding:8px; border:1px solid #ddd;'>{name}</td><td style='padding:8px; border:1px solid #ddd;'><b>{data.get('value', '')}</b> {data.get('unit', '')}</td><td style='padding:8px; border:1px solid #ddd;'>{data.get('context', '')}</td></tr>"
            else:
                html += f"<tr><td style='padding:8px; border:1px solid #ddd;'>{name}</td><td style='padding:8px; border:1px solid #ddd;' colspan='2'><b>{data}</b></td></tr>"
        html += "</table><br>"

    # Charts
    if page_data.get('charts'):
        html += "<h2 style='color: #fbbc04;'>📈 Charts Data</h2>"
        for i, chart in enumerate(page_data['charts']):
            if isinstance(chart, dict):
                html += f"<div style='background: #fff9e6; padding: 15px; margin: 10px 0; border-radius: 8px; border-left: 4px solid #fbbc04;'>"
                html += f"<h4>Chart {i+1}: {chart.get('title', 'Untitled')} ({chart.get('type', chart.get('chart_type', 'N/A'))})</h4>"

                data_dict = chart.get('data', chart.get('data_points', {}))
                if data_dict:
                    html += "<table style='width:100%; border-collapse: collapse;'>"
                    for label, val in data_dict.items():
                        html += f"<tr><td style='padding:5px; border:1px solid #ddd;'>{label}</td><td style='padding:5px; border:1px solid #ddd;'><b>{val}</b></td></tr>"
                    html += "</table>"

                if chart.get('insight'):
                    html += f"<p><i>💡 Insight: {chart['insight']}</i></p>"
                html += "</div>"

    # Tables (VLM)
    if page_data.get('tables_vlm'):
        html += "<h2 style='color: #4285f4;'>📋 Tables (VLM)</h2>"
        for i, table in enumerate(page_data['tables_vlm']):
            if isinstance(table, dict):
                html += f"<div style='margin: 10px 0;'><h4>Table {i+1}: {table.get('title', 'Untitled')}</h4>"

                headers = table.get('headers', [])
                rows = table.get('data_rows', table.get('rows', []))

                if headers or rows:
                    html += "<table style='width:100%; border-collapse: collapse;'>"
                    if headers:
                        html += "<tr style='background:#e8f0fe;'>"
                        for h in headers:
                            html += f"<th style='padding:8px; border:1px solid #ddd;'>{h}</th>"
                        html += "</tr>"
                    for row in rows[:15]:
                        html += "<tr>"
                        for cell in row:
                            html += f"<td style='padding:8px; border:1px solid #ddd;'>{cell}</td>"
                        html += "</tr>"
                    html += "</table>"
                html += "</div>"

    # Tables (OCR)
    if page_data.get('tables_ocr'):
        html += "<h2 style='color: #9334e6;'>📋 Tables (OCR Structure)</h2>"
        for i, table in enumerate(page_data['tables_ocr']):
            if isinstance(table, dict):
                html += f"<div style='margin: 10px 0;'><h4>Table {i+1}</h4>"
                html += "<table style='width:100%; border-collapse: collapse;'>"
                for row in table.get('all_data', [])[:15]:
                    html += "<tr>"
                    for cell in row:
                        html += f"<td style='padding:8px; border:1px solid #ddd;'>{cell}</td>"
                    html += "</tr>"
                html += "</table></div>"

    # Growth Rates
    if page_data.get('growth_rates'):
        html += "<h2 style='color: #0d904f;'>📈 Growth Rates</h2><ul>"
        for g in page_data['growth_rates']:
            if isinstance(g, dict):
                html += f"<li><b>{g.get('metric', '')}</b>: {g.get('rate', '')} ({g.get('period', '')})"
                if g.get('from') and g.get('to'):
                    html += f" - From {g['from']} to {g['to']}"
                html += "</li>"
        html += "</ul>"

    # Targets
    if page_data.get('targets_projections'):
        html += "<h2 style='color: #c5221f;'>🎯 Targets & Projections</h2><ul>"
        for t in page_data['targets_projections']:
            if isinstance(t, dict):
                html += f"<li><b>{t.get('target', '')}</b>: {t.get('value', '')} by {t.get('by_when', '')}</li>"
        html += "</ul>"

    # Investment Opportunities
    if page_data.get('investment_opportunities'):
        html += "<h2 style='color: #137333;'>💰 Investment Opportunities</h2><ul>"
        for inv in page_data['investment_opportunities']:
            if isinstance(inv, dict):
                html += f"<li><b>{inv.get('sector', '')}</b>: {inv.get('amount', '')} by {inv.get('timeline', '')}</li>"
        html += "</ul>"

    # Country Data
    if page_data.get('country_data'):
        html += "<h2 style='color: #1967d2;'>🌍 Country Data</h2>"
        for country, data in page_data['country_data'].items():
            html += f"<h4>{country}</h4><ul>"
            if isinstance(data, dict):
                for metric, value in data.items():
                    html += f"<li>{metric}: <b>{value}</b></li>"
            html += "</ul>"

    # Validated Numbers
    validated = page_data.get('validated_numbers', {})
    if validated.get('all_unique'):
        html += "<h2 style='color: #5f6368;'>🔢 All Extracted Numbers</h2>"
        html += f"<p style='background: #f8f9fa; padding: 15px; border-radius: 8px;'>{', '.join(validated['all_unique'][:50])}"
        if len(validated['all_unique']) > 50:
            html += f" ... and {len(validated['all_unique'])-50} more"
        html += "</p>"

    # Factual Statements
    if page_data.get('factual_statements'):
        html += "<h2 style='color: #188038;'>✅ All Factual Statements</h2>"
        html += "<div style='background: #e6f4ea; padding: 15px; border-radius: 8px; max-height: 400px; overflow-y: auto;'>"
        html += "<ol>"
        for fact in page_data['factual_statements']:
            html += f"<li style='margin: 5px 0;'>{fact}</li>"
        html += "</ol></div>"

    # All Text (OCR)
    if page_data.get('all_text_ocr'):
        html += "<h2 style='color: #5f6368;'>📝 All OCR Text</h2>"
        html += "<div style='background: #f8f9fa; padding: 15px; border-radius: 8px; max-height: 300px; overflow-y: auto; font-size: 12px;'>"
        for text in page_data['all_text_ocr']:
            html += f"{text}<br>"
        html += "</div>"

    html += "</div>"
    display(HTML(html))

print("✅ Results viewer ready")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VIEW ALL RESULTS
# ═══════════════════════════════════════════════════════════════════════════════

for page in results['pages']:
    display_comprehensive_results(page['page_number'])
    print("\n" + "═"*80 + "\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT COMPLETE DATA
# ═══════════════════════════════════════════════════════════════════════════════

# JSON export
json_file = f"{os.path.splitext(pdf_file)[0]}_FULL_EXTRACTION.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)
print(f"✅ JSON: {json_file}")

# Detailed text report
txt_file = f"{os.path.splitext(pdf_file)[0]}_DETAILED_REPORT.txt"
with open(txt_file, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("MAXIMUM ACCURACY EXTRACTION REPORT\n")
    f.write("="*80 + "\n\n")
    f.write(f"Document: {results['document_info']['source_file']}\n")
    f.write(f"Processed: {results['document_info']['timestamp']}\n")
    f.write(f"Pipeline: {results['document_info']['pipeline_version']}\n\n")

    for page in results['pages']:
        f.write("\n" + "="*80 + "\n")
        f.write(f"SLIDE {page['page_number']}: {page.get('slide_title', 'Untitled')}\n")
        f.write(f"Type: {page.get('slide_type', 'N/A')} | Quality: {page.get('extraction_quality', 'N/A').upper()}\n")
        f.write("="*80 + "\n\n")

        # Key highlights
        if page.get('key_highlights'):
            f.write("KEY HIGHLIGHTS:\n")
            for h in page['key_highlights']:
                f.write(f"  • {h}\n")
            f.write("\n")

        # Metrics
        if page.get('metrics'):
            f.write("METRICS:\n")
            for name, data in page['metrics'].items():
                if isinstance(data, dict):
                    f.write(f"  • {name}: {data.get('value', '')} {data.get('unit', '')}\n")
                else:
                    f.write(f"  • {name}: {data}\n")
            f.write("\n")

        # Charts
        if page.get('charts'):
            f.write(f"CHARTS ({len(page['charts'])}):\n")
            for chart in page['charts']:
                if isinstance(chart, dict):
                    f.write(f"  Chart: {chart.get('title', 'Untitled')}\n")
                    for label, val in chart.get('data', chart.get('data_points', {})).items():
                        f.write(f"    - {label}: {val}\n")
            f.write("\n")

        # All factual statements
        if page.get('factual_statements'):
            f.write(f"FACTUAL STATEMENTS ({len(page['factual_statements'])}):\n")
            for i, fact in enumerate(page['factual_statements'], 1):
                f.write(f"  {i}. {fact}\n")
            f.write("\n")

        # All numbers
        validated = page.get('validated_numbers', {})
        if validated.get('all_unique'):
            f.write(f"ALL NUMBERS ({len(validated['all_unique'])}):\n")
            f.write(f"  {', '.join(validated['all_unique'])}\n\n")

print(f"✅ Report: {txt_file}")

# Download
from google.colab import files
files.download(json_file)
files.download(txt_file)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# QUICK DATA SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*80)
print("📊 COMPLETE DATA EXTRACTION SUMMARY")
print("═"*80)

for page in results['pages']:
    print(f"\n▸ SLIDE {page['page_number']}: {page.get('slide_title', 'Untitled')[:60]}")
    print(f"  Quality: {page.get('extraction_quality', 'N/A').upper()}")
    print(f"  Facts extracted: {len(page.get('factual_statements', []))}")
    print(f"  Numbers found: {len(page.get('validated_numbers', {}).get('all_unique', []))}")

    # Print top facts
    print("  Top Data Points:")
    for fact in page.get('factual_statements', [])[:8]:
        print(f"    ✓ {fact[:80]}")

print("\n" + "═"*80)